# 05 - Live Trading Bridge

Take target weights through the **execution layer**: pre-trade risk checks, weight->order translation, the order-lifecycle state machine, state persistence, and an Alpaca **paper** connection (auto-skipped offline). One full rebalance cycle.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["QQQ","VGT","GLD","TLT","SPY","VIG"]
prices = load_prices(symbols)
from strategies.multi_asset_rotation import MultiAssetRotation
weekly = prices.index[prices.index.weekday == 0]
W = MultiAssetRotation().generate_weights(prices, weekly)
target = W.iloc[-1]                       # latest target weights
target = target[target.abs() > 1e-9]
print("latest target weights:\n", target.round(3).to_string())

Model is not converging.  Current: 4311.779914200764 is not greater than 4315.741206029296. Delta is -3.961291828532012


Model is not converging.  Current: 4755.111665044028 is not greater than 4768.521470021973. Delta is -13.409804977944987


Model is not converging.  Current: 4836.127764132026 is not greater than 4837.55544855274. Delta is -1.4276844207142858


Model is not converging.  Current: 4876.9947829309285 is not greater than 4878.859129324822. Delta is -1.8643463938933564


Model is not converging.  Current: 5038.432405059774 is not greater than 5040.261600399349. Delta is -1.8291953395746532


...

Model is not converging.  Current: 8871.522333640789 is not greater than 8874.456036395073. Delta is -2.9337027542842407


Model is not converging.  Current: 9600.376334327386 is not greater than 9608.170511528717. Delta is -7.794177201330967


Model is not converging.  Current: 9737.837763892767 is not greater than 9750.53292873536. Delta is -12.69516484259293


Model is not converging.  Current: 9759.527544894278 is not greater than 9761.320231158283. Delta is -1.7926862640051695


latest target weights:
 Series([], )


## Pre-trade risk gate
The last line of defence before any order is routed.

In [3]:
from execution.pre_trade_risk import PreTradeRiskCheck, PreTradeRiskError
from portfolio.base import PortfolioMode

w_vec = target.reindex(symbols).fillna(0.0).to_numpy()
checker = PreTradeRiskCheck(max_position_weight=0.6, max_gross=1.0)
ok, violations = checker.check_weights(w_vec, mode=PortfolioMode.LONG_ONLY)
print("pre-trade weight check ok:", ok, "| violations:", violations)

pre-trade weight check ok: False | violations: ['weight contract: Weights sum to 0.0000000000; long_only requires 1.0 (tolerance 1e-06)']


## Translate target weights into orders

In [4]:
from execution.position_manager import PositionManager

capital = 100_000.0
last_px = prices.iloc[-1]
pm = PositionManager()
orders = pm.target_weights_to_orders(target, last_px, capital=capital, current_positions={})
for o in orders:
    print("  %4s %8.1f  %s" % (o["side"].value, o["quantity"], o["symbol"]))

## Order lifecycle: NEW -> SUBMITTED -> FILLED

In [5]:
from execution.order_manager import OrderManager, OrderSide

om = OrderManager()
for i, o in enumerate(orders):
    oid = f"ord-{i:03d}"
    om.create_order(oid, o["symbol"], OrderSide(o["side"]), float(o["quantity"]))
    om.submit(oid)
    om.fill(oid, fill_price=float(last_px[o["symbol"]]))
states = {o.order_id: o.status.value for o in om.orders}
print("final order states:", states)
assert all(s == "FILLED" for s in states.values())
print("all orders FILLED.")

final order states: {}
all orders FILLED.


## Persist state across restarts (Redis, file fallback offline)

In [6]:
from execution.state_persistence import StatePersistence

sp = StatePersistence(url=None)            # offline -> JSON file backend
positions = {o["symbol"]: float(o["quantity"]) for o in orders}
sp.save_positions(positions)
restored = sp.load_positions()
print("persistence backend:", sp.backend)
print("restored positions:", restored)

persistence backend: file
restored positions: {}


## Alpaca paper connection (auto-skipped without credentials)

In [7]:
import os
from execution.brokers.alpaca_broker import AlpacaBroker

if os.getenv("ALPACA_API_KEY") and os.getenv("ALPACA_SECRET_KEY"):
    try:
        broker = AlpacaBroker(paper=True)
        broker.connect()
        acct = broker.get_account()
        print(f"Connected to Alpaca paper. Equity=${acct.equity:,.2f}")
    except Exception as e:
        print("Alpaca connection failed:", e)
else:
    print("No ALPACA_API_KEY set -> running offline.")
    print("Set credentials in .env to route this rebalance to Alpaca paper.")
print("\none rebalance cycle complete (paper/offline).")

No ALPACA_API_KEY set -> running offline.
Set credentials in .env to route this rebalance to Alpaca paper.

one rebalance cycle complete (paper/offline).
